## Q0. Setup

In [1]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(f"loaded {len(documents)} documents")   # should print 72

# We'll also need chunks for search later
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"{len(chunks)} chunks")

loaded 72 documents
295 chunks


In [2]:
import os
import httpx
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("BASE_URL"),
    http_client=httpx.Client(verify=False),
)

In [3]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
import json
from bacterio_compat import llm_structured

doc = documents[0]
user_prompt = json.dumps({"filename": doc["filename"], "content": doc["content"]})

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions,
)

print(f"filename: {doc['filename']}\n")
for q in result.questions:
    print(f"  - {q}")
print(f"\ntokens: in={usage.prompt_tokens}, out={usage.completion_tokens}")

filename: 01-agentic-rag/lessons/01-intro.md

  - What exactly does RAG do to help with the problems LLMs have?
  - Do I need to know how to host an LLM on my own hardware for this course?
  - Can you explain what an LLM is basically doing when it generates text?
  - What's the main goal of Part 2 compared to Part 1 of this module?
  - Are we going to use a framework to build the RAG system, or do we do it manually?

tokens: in=1196, out=911


## Q1. Generating questions.

In [5]:
import json

# The first 3 lesson pages we want to process
target_filenames = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

# Filter to just those documents
target_docs = [d for d in documents if d["filename"] in target_filenames]
print(f"matched {len(target_docs)} documents\n")

# Generate questions for each and collect token usage
input_token_counts = []

for doc in target_docs:
    user_prompt = json.dumps({"filename": doc["filename"], "content": doc["content"]})
    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
    )
    print(f"{doc['filename']}: input={usage.prompt_tokens}, output={usage.completion_tokens}")
    input_token_counts.append(usage.prompt_tokens)

avg_input = sum(input_token_counts) / len(input_token_counts)
print(f"\nAverage input tokens: {avg_input:.0f}")

matched 3 documents

01-agentic-rag/lessons/01-intro.md: input=1196, output=953
01-agentic-rag/lessons/02-environment.md: input=1533, output=965
01-agentic-rag/lessons/03-rag.md: input=2019, output=951

Average input tokens: 1583


## Ground Truth

In [6]:
import subprocess
import pandas as pd

subprocess.run([
    "wget", "-nc",
    "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv"
], check=False)

df = pd.read_csv("ground-truth.csv")
ground_truth = df.to_dict(orient="records")
print(f"loaded {len(ground_truth)} ground truth records")
print("first record:", ground_truth[0])

loaded 360 ground truth records
first record: {'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?", 'filename': '01-agentic-rag/lessons/01-intro.md'}


File ‘ground-truth.csv’ already there; not retrieving.



In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"{len(chunks)} chunks") #295

295 chunks


In [10]:
from minsearch import Index, VectorSearch
import numpy as np
from embedder import Embedder
embedder = Embedder()

# Text index (keyword search)
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

texts = [c["content"] for c in chunks]
X = np.asarray(embedder.encode_batch(texts))

vector_index = VectorSearch()
vector_index.fit(X, chunks)

print("indexes ready")

indexes ready


In [11]:
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

def vector_search(query, num_results=5):
    q_vec = np.asarray(embedder.encode(query)).flatten()
    return vector_index.search(q_vec, num_results=num_results)

In [12]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

## Q2. First result with text search

In [13]:
q = ground_truth[0]["question"]
print(f"query: {q}\n")

results = text_search(q)
print("text_search top 5:")
for r in results:
    print(f"  {r['filename']}  start={r['start']}")

print(f"\nAnswer to Q2: {results[0]['filename']}")

query: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?

text_search top 5:
  01-agentic-rag/lessons/03-rag.md  start=3000
  01-agentic-rag/lessons/13-function-calling.md  start=1000
  01-agentic-rag/lessons/03-rag.md  start=2000
  01-agentic-rag/lessons/13-function-calling.md  start=2000
  01-agentic-rag/lessons/01-intro.md  start=0

Answer to Q2: 01-agentic-rag/lessons/03-rag.md


## Q3. First result with vector search

In [14]:
q = ground_truth[0]["question"]
print(f"query: {q}\n")

results = vector_search(q)
print("vector_search top 5:")
for r in results:
    print(f"  {r['filename']}  start={r['start']}")

print(f"\nAnswer to Q3: {results[0]['filename']}")

query: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?

vector_search top 5:
  01-agentic-rag/lessons/01-intro.md  start=0
  04-evaluation/lessons/11-evaluation-intro.md  start=0
  04-evaluation/lessons/12-rag-answers.md  start=2000
  01-agentic-rag/lessons/10-rag-next-steps.md  start=0
  06-best-practices/lessons/01-intro.md  start=0

Answer to Q3: 01-agentic-rag/lessons/01-intro.md


## Q4. Evaluating text search

In [15]:
from tqdm.auto import tqdm

def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(q["question"])
    return [int(r["filename"] == filename) for r in results]

def hit_rate(relevance):
    return sum(1 for line in relevance if 1 in line) / len(relevance)

def mrr(relevance):
    total = 0.0
    for line in relevance:
        for rank, val in enumerate(line):
            if val == 1:
                total += 1 / (rank + 1)
                break
    return total / len(relevance)

def evaluate(ground_truth, search_function):
    relevance = []
    for q in tqdm(ground_truth):
        relevance.append(compute_relevance(q, search_function))
    return {"hit_rate": hit_rate(relevance), "mrr": mrr(relevance)}

# Run it for text search
text_metrics = evaluate(ground_truth, text_search)
print(text_metrics)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}


## Q5. Vector evaluation

In [16]:
vector_metrics = evaluate(ground_truth, vector_search)
print(vector_metrics)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}


## Q6. Tune k for hybrid search

In [17]:
results = []

for k in [1, 50, 100, 200]:
    print(f"Evaluating k={k}...")
    # capture k at lambda-definition time, not at call time
    metrics = evaluate(
        ground_truth,
        lambda query, k=k: hybrid_search(query, k=k),
    )
    results.append({"k": k, **metrics})

import pandas as pd
df = pd.DataFrame(results)
print(df.sort_values("mrr", ascending=False))

Evaluating k=1...


  0%|          | 0/360 [00:00<?, ?it/s]

Evaluating k=50...


  0%|          | 0/360 [00:00<?, ?it/s]

Evaluating k=100...


  0%|          | 0/360 [00:00<?, ?it/s]

Evaluating k=200...


  0%|          | 0/360 [00:00<?, ?it/s]

     k  hit_rate       mrr
0    1  0.838889  0.648194
1   50  0.836111  0.637917
2  100  0.836111  0.637917
3  200  0.836111  0.637917
